Cài đặt thư viện

In [ ]:
!pip install -q transformers accelerate tiktoken torchvision
!pip install -q Levenshtein pandas tqdm

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import Levenshtein

from pathlib import Path
from tqdm import tqdm
from PIL import Image
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Đường dẫn
ZIP_PATH = Path('/content/drive/MyDrive/IntroToML - OCR - data/processed_data.zip')
LOCAL_ROOT = Path('/content/local_data')
LOCAL_EXTRACTED_DIR = LOCAL_ROOT / 'processed_data'

# 3. Giải nén (Giữ nguyên logic cực nhanh này)
if ZIP_PATH.exists():
    if not LOCAL_EXTRACTED_DIR.exists():
        print(f"--- Đang giải nén {ZIP_PATH.name} vào máy ảo ---")
        !unzip -q "{ZIP_PATH}" -d "{LOCAL_ROOT}"
        print("-> Giải nén hoàn tất!")
    else:
        print("-> Dữ liệu đã có sẵn trên máy ảo Colab, bỏ qua giải nén.")
else:
    print(f"❌ LỖI: Không tìm thấy file {ZIP_PATH}")

TEST_DIR = LOCAL_EXTRACTED_DIR / 'test'
print(f"\n Đường dẫn thư mục Test: {TEST_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[*] Dang quet du lieu trong: /content/drive/MyDrive/IntroToML - OCR/Data (small)/train
[*] Tong so anh hop le: 50
[*] Thiet bi su dung: cuda | Batch size: 8


Các hàm chức năng

In [ ]:
def normalize_text(text: str) -> str:
    """Chuẩn hóa văn bản tiếng Việt: Đưa về chuẩn NFC và viết thường."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()

def calculate_cer(pred: str, gt: str) -> float:
    """Tính tỉ lệ lỗi ký tự (CER)."""
    pred = normalize_text(pred)
    gt = normalize_text(gt)

    if len(gt) == 0:
        return 1.0 if len(pred) > 0 else 0.0

    edit_dist = Levenshtein.distance(pred, gt)
    cer = edit_dist / len(gt)
    return cer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Dang tai base model: zai-org/GLM-OCR (GlmOcrForConditionalGeneration)...


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Tai base model thanh cong.



Cài đặt mô hình

In [ ]:
# ==========================================
# 1. KHỞI TẠO MÔ HÌNH GLM-OCR Ở ĐÂY
# (Hãy dán đoạn code tải model, tokenizer của bạn vào khu vực này)
# Ví dụ:
# model = AutoModelForCausalLM.from_pretrained(...)
# tokenizer = AutoTokenizer.from_pretrained(...)
# ==========================================
print("Đang nạp cấu hình Test cho GLM-OCR...")

# 2. Quét qua tất cả thư mục con
all_test_samples = []
subfolders = [f for f in TEST_DIR.iterdir() if f.is_dir()]

for subfolder in subfolders:
    label_file = subfolder / 'label.json'
    if not label_file.exists():
        continue

    with open(label_file, 'r', encoding='utf-8') as f:
        ground_truths = json.load(f)

    for img_name, gt_text in ground_truths.items():
        img_path = subfolder / img_name
        if img_path.exists():
            all_test_samples.append((img_path, gt_text))

print(f"Thực hiện đánh giá trên {len(all_test_samples)} ảnh \n")

# 3. Vòng lặp đánh giá
total_cer = 0.0
error_logs = []

# PROMPT TỐI ƯU CHO GLM-OCR (Ngăn nó nói nhảm như "Đây là đoạn văn bản trong ảnh:...")
SYSTEM_PROMPT = "You are a highly accurate OCR system. Extract the text from the image exactly as it is written. Output ONLY the extracted text, without any conversational filler, markdown formatting, or additional explanations."

for img_path, gt_text in tqdm(all_test_samples, desc="Đang suy luận GLM-OCR"):
    try:
        # ==========================================
        # 4. THỰC HIỆN DỰ ĐOÁN VỚI GLM-OCR
        # (Thay thế đoạn try này bằng hàm sinh text của mô hình bạn dùng)
        # ==========================================
        img = Image.open(str(img_path)).convert('RGB')

        # --- ĐOẠN CODE MẪU (Tùy thuộc vào thư viện GLM bạn dùng) ---
        # response, _ = model.chat(tokenizer, image=img, query=SYSTEM_PROMPT)
        # pred_text = response

        # Lưu ý: Cần chắc chắn biến `pred_text` chứa chuỗi text sau khi nhận diện
        pred_text = "..." # Thay bằng kết quả thật của GLM

    except Exception as e:
        # Nếu model báo lỗi (quá tải VRAM, ảnh hỏng...), trả về chuỗi rỗng
        pred_text = ""

    # Tính điểm CER
    cer_score = calculate_cer(pred_text, gt_text)
    total_cer += cer_score

    # Ghi log ảnh lỗi
    if cer_score > 0:
        error_logs.append({
            "folder": img_path.parent.name,
            "image": img_path.name,
            "ground_truth": normalize_text(gt_text),
            "prediction": normalize_text(pred_text),
            "cer_score": round(cer_score, 4)
        })

# Tính toán CER trung bình
test_samples = len(all_test_samples)
average_cer = total_cer / test_samples if test_samples > 0 else 0

print(f"\nCER trung bình (GLM-OCR): {average_cer * 100:.2f} %")
print(f"Số lượng ảnh dự đoán sai: {len(error_logs)}")

In [ ]:
# ==============================================================================
# 4. CHAY DANH GIA
# ==============================================================================

print("\n[*] Bat dau danh gia...")

evaluation_results = []
hard_negatives = []
total_cer = 0.0
successful_evals = 0
NUM_WORKERS = 2
EVAL_BATCH_SIZE = 16

test_loader = DataLoader(
    OCRDataset(all_test_items),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn,
    prefetch_factor=2,
)

for batch_idx, (batch_items, batch_images) in enumerate(
    tqdm(test_loader, desc="Evaluating")
):
    if batch_idx > 0 and batch_idx % 50 == 0:
        torch.cuda.empty_cache()

    try:
        predictions = run_ocr_batch_fast(base_model, base_processor, batch_images)

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\n⚠️ OOM tại batch {batch_idx} → fallback từng ảnh...")
            torch.cuda.empty_cache()
        else:
            print(f"\n⚠️ Lỗi batch {batch_idx}: {e}")

        # Fallback: chạy từng ảnh một để tránh OOM hoàn toàn
        predictions = []
        for img in batch_images:
            try:
                pred = run_ocr_batch_fast(base_model, base_processor, [img])
                predictions.extend(pred)
            except Exception:
                predictions.append("")
                torch.cuda.empty_cache()

    for item, pred in zip(batch_items, predictions):
        gt = item["ground_truth"]
        cer_score = _cer(gt, pred)

        total_cer += cer_score
        successful_evals += 1

        record = {
            "folder": item["folder"],
            "img_name": item["img_name"],
            "ground_truth": gt,
            "prediction": pred,
            "cer": round(cer_score, 4),
        }
        evaluation_results.append(record)

        if cer_score > 0.5:
            hard_negatives.append(record)

average_cer = total_cer / successful_evals if successful_evals else 0.0

print("\n" + "=" * 50)
print("KET QUA")
print("=" * 50)
print(f"So anh da danh gia : {successful_evals}")
print(f"Average CER        : {average_cer:.4f} ({average_cer * 100:.2f}%)")
print(f"Hard negatives     : {len(hard_negatives)}")

print("\n--- XEM THỬ CÁC MẪU LỖI NẶNG (Hard Negatives) ---")
# In ra 5 mẫu lỗi nặng nhất để phân tích
for bad_case in hard_negatives[:5]:
    print(f"File: {bad_case['img_name']}")
    print(f"CER: {bad_case['cer']}")
    print(f"Ground Truth : {bad_case['ground_truth']}")
    print(f"Prediction   : {bad_case['prediction']}")
    print("-" * 50)


[*] Bat dau danh gia...


Evaluating: 100%|██████████| 4/4 [00:11<00:00,  2.82s/it]


KET QUA
So anh da danh gia : 50
Average CER        : 0.5157 (51.57%)
Hard negatives     : 13

--- XEM THỬ CÁC MẪU LỖI NẶNG (Hard Negatives) ---
File: 4.jpg
CER: 0.6667
Ground Truth : đơn
Prediction   : đien
--------------------------------------------------
File: 6.jpg
CER: 1.0
Ground Truth : là
Prediction   : lai
--------------------------------------------------
File: 13.jpg
CER: 6.0
Ground Truth : ở
Prediction   : 0 ^{2}
--------------------------------------------------
File: 18.jpg
CER: 1.0
Ground Truth : km2
Prediction   : (
--------------------------------------------------
File: 23.jpg
CER: 1.0
Ground Truth : đặc
Prediction   : clac
--------------------------------------------------


Xuất báo cáo

In [ ]:
# Chuyển đổi danh sách lỗi thành Pandas DataFrame
df_errors = pd.DataFrame(error_logs)

if not df_errors.empty:
    # Sắp xếp từ lỗi nặng nhất xuống nhẹ nhất
    df_errors = df_errors.sort_values(by="cer_score", ascending=False)

    # Lưu file nội bộ
    report_path = LOCAL_ROOT / 'baseline_GLMOCR_report.csv'
    df_errors.to_csv(report_path, index=False, encoding='utf-8-sig')

    # Copy sang Google Drive
    drive_report_path = Path('/content/drive/MyDrive/IntroToML - OCR - data/baseline_GLMOCR_report.csv')
    !cp "{report_path}" "{drive_report_path}"

    print(f"Đã lưu lại report của baseline_GLMOCR tại Drive.")
else:
    print("Mô hình GLM-OCR dự đoán đúng hoàn hảo 100%!")